# 16 | Agent ReAct:看一次完整的思考、行动和观察过程

Agent ReAct 讲的是一件很具体的事：**模型不要直接拍结论，而是边判断、边调用工具、边根据结果调整下一步。**

ReAct 可以简单理解成三步：

```text
Reason 判断需要什么 -> Action 调用工具 -> Observe 读取结果
```

比如用户问“订单能不能退款”，Agent 不能张口就答。它得先查订单状态，再查退款规则，必要时创建跟进工单。
本文就用这个售后场景来举例。

## 一、准备模型和工具

这个例子想说明一件事：**Agent 不是直接回答退款结论，而是先把问题拆成几个可执行动作。**

我们准备一个支持 tool calling 的模型，再准备三个售后工具：查订单状态、查退款规则、创建售后跟进工单。

真实项目里，这些工具背后可以是数据库、HTTP API、CRM 或工单系统。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

# 显式读取当前项目下的 .env，避免不同运行环境下自动查找路径不一致。
load_dotenv(dotenv_path=".env")

# 这里使用 LangChain 推荐的 init_chat_model 统一创建模型。
# 只要底层模型支持 tool calling，Agent 才能稳定调用工具。
model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.environ.get("DASHSCOPE_BASE_URL"),
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
)


In [ ]:
from langchain.tools import tool

# 模拟业务数据：真实项目里通常来自数据库、知识库或业务系统 API。
ORDERS = {
    "O-1001": {"item": "蓝牙耳机", "status": "未发货"},
    "O-1002": {"item": "机械键盘", "status": "已发货"},
}

POLICIES = {
    "未发货": "可以直接退款，系统会拦截发货，1 到 3 个工作日原路退回。",
    "已发货": "需要先退货，仓库验收通过后，3 到 7 个工作日原路退回。",
}

TICKETS = []


@tool
def get_order_status(order_id: str) -> dict:
    """根据订单号查询订单状态。"""
    # @tool 会把函数暴露给 Agent，函数名、参数和 docstring 都会影响模型选择。
    order = ORDERS.get(order_id)
    return {"order_id": order_id, **order} if order else {"error": "订单不存在"}


@tool
def get_refund_policy(order_status: str) -> str:
    """根据订单状态查询退款规则。"""
    # 规则由工具返回，模型负责基于规则组织回答。
    return POLICIES.get(order_status, "没有匹配规则，需要转人工。")


@tool
def create_refund_followup_ticket(order_id: str, issue: str) -> dict:
    """创建退款跟进工单。"""
    # 用最小字段模拟工单创建结果。
    ticket = {"ticket_id": f"TK-{len(TICKETS) + 1:04d}", "order_id": order_id, "issue": issue}
    TICKETS.append(ticket)
    return ticket


tools = [get_order_status, get_refund_policy, create_refund_followup_ticket]


## 二、创建 Agent

`create_agent` 会把模型、工具和系统提示词组装成一个 Agent。

这里的系统提示词负责约束工作顺序：先查订单，再查规则；只有用户明确要求跟进时，才创建工单。顺序说清楚，Agent 少走弯路，用户少血压升高。

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=tools,
    # system_prompt 用来约束 Agent 的工作顺序，避免它跳过查询直接回答。
    system_prompt=(
        "你是一个电商售后 Agent。"
        "处理退款问题时，不能凭空判断订单状态；必须先调用 get_order_status。"
        "拿到订单状态后，再调用 get_refund_policy 查询退款规则。"
        "只有当用户明确要求跟进、登记或创建工单时，才调用 create_refund_followup_ticket。"
        "最终回答要简洁说明：是否能退、依据是什么、预计多久到账、是否已创建工单。"
    ),
)


## 三、先看最终回答

最普通的调用方式是 `invoke()`。它会跑完整个 Agent 循环，然后返回最终状态。

这适合业务调用，但只看最终答案，看不到中间到底发生了什么。

In [ ]:
question = "用户说：订单 O-1001 想退款，请判断是否能退、多久到账，并创建一个售后跟进工单。"

# invoke 会执行完整 Agent 流程，但这里只拿最终结果。
response = agent.invoke({
    "messages": [
        {"role": "user", "content": question}
    ]
})

# 最后一条消息通常就是 Agent 给用户的最终回答。
print(response["messages"][-1].content)


## 四、打印 ReAct 轨迹

为了看到中间过程，可以用 `stream()` 边执行边读取消息状态。

下面这个函数会打印四类信息：

- `Thought`：可见决策摘要，也就是模型准备调用什么工具
- `Action`：工具名和入参
- `Observation`：工具返回结果
- `Final Answer`：最终回答

这里的 `Thought` 不是模型隐藏推理原文，而是根据工具调用整理出来的可观察决策。工程调试看这个就够了，也更安全。

In [ ]:
from langchain.messages import AIMessage, HumanMessage, ToolMessage


def print_react_trace(agent, question: str):
    """流式执行 Agent，并打印可观察的 ReAct 轨迹。"""
    seen = set()

    # stream 比 invoke 更适合观察中间过程。
    for state in agent.stream({"messages": [{"role": "user", "content": question}]}, stream_mode="values"):
        for message in state["messages"]:
            message_id = getattr(message, "id", None) or repr(message)
            if message_id in seen:
                continue
            seen.add(message_id)

            if isinstance(message, HumanMessage):
                print(f"User: {message.content}\n")

            elif isinstance(message, AIMessage) and message.tool_calls:
                # tool_calls 就是模型决定采取的 Action。
                for call in message.tool_calls:
                    print(f"Thought: 需要调用工具 {call['name']}")
                    print(f"Action Input: {call['args']}\n")

            elif isinstance(message, ToolMessage):
                # ToolMessage 是工具返回的 Observation。
                print(f"Observation: {message.content}\n")

            elif isinstance(message, AIMessage) and message.content:
                print(f"Final Answer: {message.content}\n")


现在重新问一次同样的问题，这次不只看答案，而是看它怎么一步步做事。

In [ ]:
# 这次不只看最终回答，而是观察完整的 ReAct 轨迹。
print_react_trace(agent, question)


一次正常的输出会接近上面这样，这就是 ReAct 最值得看的地方：它不是一次性把答案写出来，而是每拿到一个观察结果，再决定下一步。

再换一个已发货订单，并且明确说“不用建工单”，看看 Agent 会不会调整路径。

In [ ]:
another_question = "订单 O-1002 想退款，不用建工单。请告诉我现在能不能直接退，以及大概多久到账。"

# 换一个订单和诉求，观察 Agent 是否会调整工具调用路径。
print_react_trace(agent, another_question)
